# Phutball automatic curriculum (expert iteration, Qwen3.5-9B thinking on, board + men list)

Every round runs by fixed rules (llm_train/curriculum.py), so the loop runs unattended:
- **allocate** 300 puzzles from the pool across buckets (1-4-jump wins, backward-trap wins, no-win, block, forced,
  prevent) by expert-iteration priority (1-(1-p)^4)(1-p) of each bucket's solve rate: most compute where the model
  sometimes succeeds;
- **parole** puzzles with no verified solution in their last 2 attempts; **release** them when their bucket's rate rises
  15 points, or when a 5% spot check solves them;
- **sample** 4 thinking rollouts per puzzle with the current best model, keep engine-verified solutions (random choice,
  natural finishes first), **balance** win and no-win examples 1:1, fine-tune from the base model;
- **gate** (every round, on a light eval: 25 items per group, 1 sample): the new model generates the next round only if
  its mean finished pass@1 beats the best so far and no family's pooled forced accuracy (wins, no-win, placements)
  drops more than 10 points; otherwise the next round samples from the previous best. A full eval (4 samples) runs on
  round 1 and every 5th round for the curves.

Prompt format **ascii+men** (board plus the list of men) everywhere, including the baselines. Starts from the
self-stop warm start. **Runtime:** A100, High-RAM. **Cost:** baselines ~1.2 h; each round ~1 h (~7 units); 30 rounds ~30 h (~210 units).
Resumable: everything is in Drive; re-run all cells after a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, subprocess, time, urllib.request
OUT = '/content/drive/MyDrive/phutball/curriculum'; os.makedirs(OUT, exist_ok=True)
WARM = '/content/drive/MyDrive/phutball/grpo_pilot/self_stop/adapter/final'
POOL = 'llm_train/puzzles/pool_v2.jsonl.gz'; STATE = f'{OUT}/state.json'; FMT = 'ascii+men'
CAP = 6144; N = 300; K = 4; ROUNDS = 30            # ~1 h and ~7 units per round; raise ROUNDS any time (resumes)
PG = 25; EK = 4                                     # full eval (curves): 25 items per group x 4 samples
GATE_EK = 1; FULL_EVERY = 5                         # gate eval every round: 25 per group x 1 sample; full eval on round 1 and every 5th
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention
!pip -q uninstall -y torchao
!python -c "import trl, vllm, peft, transformers; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__)"

In [ ]:
def serve(adapters: dict, max_len=CAP + 2048):
    !pkill -f "vllm serve" ; sleep 5
    extra = ['--enable-lora', '--max-lora-rank', '32', '--max-loras', str(max(1, len(adapters))), '--lora-modules',
             *[f'{k}={v}' for k, v in adapters.items()]] if adapters else []
    log = open('/content/vllm_serve.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3.5-9B', '--max-model-len', str(max_len), '--gpu-memory-utilization',
                          '0.90', '--port', '8000', *extra], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        if p.poll() is not None: break
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up:', list(adapters) or 'base'); return p
        except Exception: time.sleep(5)
    !tail -15 /content/vllm_serve.log
    raise RuntimeError('vLLM did not start')

EV = f'{OUT}/eval'; os.makedirs(EV, exist_ok=True)
def evaluate(name, model, samples=EK):
    if os.path.exists(f'{EV}/{name}-forced.summary.json'): print('done:', name); return
    !python -m llm_bench.run --model {model} --name {name} --prompt-format {FMT} --thinking true --qwen-thinking-sampling --samples {samples} --max-tokens {CAP} --per-group {PG} --concurrency 96 | tail -10
    !python -m llm_bench.run --model {model} --tokenizer Qwen/Qwen3.5-9B --prompt-format {FMT} --thinking true --salvage {name} --per-group {PG} --concurrency 48 | tail -10
    !cp llm_bench/results/{name}* {EV}/
def best_adapter():
    return (json.load(open(STATE)).get('best') if os.path.exists(STATE) else None) or WARM

## 1. Baselines with the men list (untrained, warm start): the yardstick and the gate's starting point

In [ ]:
if not all(os.path.exists(f'{EV}/{n}-forced.summary.json') for n in ('untrained', 'warmstart', 'g0')):
    server = serve({'warmstart': WARM})
    evaluate('untrained', 'Qwen/Qwen3.5-9B'); evaluate('warmstart', 'warmstart')
    evaluate('g0', 'warmstart', samples=GATE_EK)          # the warm start under the gate's protocol: the first 'best'
    !pkill -f "vllm serve"
else: print('baselines done')

## 2. Curriculum rounds (unattended)

In [ ]:
for R in range(1, ROUNDS + 1):
    d = f'{OUT}/r{R}'; os.makedirs(d, exist_ok=True)
    if not os.path.exists(f'{d}/ids.json'):
        !python -m llm_train.curriculum allocate --state {STATE} --pool {POOL} --n {N} --round {R} --out {d}/ids.json | grep -E '"(alloc|spot_checks)"' -A14 | head -16
    if not os.path.exists(f'{d}/sft.jsonl'):
        print(f'=== round {R}: sampling from', best_adapter())
        !python -m llm_train.expert_iter --arm iid --adapter {best_adapter()} --pool {POOL} --ids {d}/ids.json --out {d} --k {K} --max-think {CAP} --seed {R} --prompt-format {FMT} --max-neg-ratio 1.0 2>&1 | grep --line-buffered -v -E "it/s\]|Processed prompts|Adding requests|INFO|WARNING"
    if not os.path.exists(f'{d}/adapter/final/adapter_model.safetensors'):
        !python -m llm_train.sft_stop --data {d}/sft.jsonl --out {d}/adapter --max-neg-ratio 1.0 2>&1 | grep --line-buffered -E "examples|'loss'|saved|rror|Traceback"
    full = (R == 1 or R % FULL_EVERY == 0)
    if not os.path.exists(f'{EV}/g{R}-forced.summary.json') or (full and not os.path.exists(f'{EV}/r{R}-forced.summary.json')):
        server = serve({f'r{R}': f'{d}/adapter/final'})
        evaluate(f'g{R}', f'r{R}', samples=GATE_EK)
        if full: evaluate(f'r{R}', f'r{R}')
        !pkill -f "vllm serve"
    if R not in (json.load(open(STATE)).get('applied', []) if os.path.exists(STATE) else []):
        !python -m llm_train.curriculum update --state {STATE} --round {R} --outcomes {d}/outcomes.jsonl | tail -20
    if not os.path.exists(f'{d}/gate.txt'):
        !python -m llm_train.curriculum gate --state {STATE} --round {R} --new {EV}/g{R} --adapter {d}/adapter/final --init-best {EV}/g0 --init-adapter {WARM} | tee {d}/gate.txt
print('curriculum done; best:', best_adapter())

## 3. Results

In [ ]:
names = ['untrained', 'warmstart'] + [f'r{R}' for R in range(1, ROUNDS + 1)]   # full evals (round 1 and every FULL_EVERY)
keys = ('win: positive, 1-jump', 'win: positive, 2-jump', 'win: positive, 3-jump', 'win: positive, 4-jump',
        'win: near-miss negative', 'forced placement', 'block placement', 'prevent placement')
for label, suffix in (('FINISHED ON ITS OWN', '.summary.json'), ('BUDGET-FORCED', '-forced.summary.json')):
    S = {n: json.load(open(f'{EV}/{n}{suffix}')) for n in names if os.path.exists(f'{EV}/{n}{suffix}')}
    print(f"\n{label}: pass@1 / pass@{EK}  (ascii+men, thinking on, sampled, {CAP} tokens)")
    print(f"{'':26s}" + ''.join(f"{n:>14s}" for n in S))
    for k in keys:
        print(f"{k:26s}" + ''.join(f"{S[n][k]['accuracy']:>8.0%} /{S[n][k].get(f'pass@{EK}', float('nan')):>4.0%}" if k in S[n] else f"{'-':>14s}" for n in S))
    print(f"{'cut off (wins)':26s}" + ''.join(f"{S[n].get('win (all)', {}).get('truncated', 0):>14.0%}" for n in S))
    print(f"{'mean tokens':26s}" + ''.join(f"{S[n].get('mean_completion_tokens', 0):>14.0f}" for n in S))
if os.path.exists(STATE):
    st = json.load(open(STATE))
    print('\nscheduler log:')
    for e in st['log']:
        if e['event'] == 'gate': print(f"  round {e['round']}: {'PROMOTED' if e['promoted'] else 'kept best'} ({e['reason']})")
        elif e['event'] == 'update': print(f"  round {e['round']}: rates {e['rates']} | paroled {e['paroled_now']}, released {e['released']}, {e['status']}")

In [ ]:
# optional: stop billing when everything above has finished (queue this cell last)
# from google.colab import runtime; runtime.unassign()